In [ ]:
import geopandas as gpd
import pandas as pd

Source files were put together by Kaylyn and were accessed from: `CIROH Basecamp/Projects/XXXX - MMM/GIS/RFC_Shps`

In [ ]:
basins_gdf = gpd.read_parquet("rfc_basins.parquet")
basins_gdf.head()

In [ ]:
# There are several duplicate basin_ids across RFCs, will keep these for now
basins_gdf[basins_gdf.duplicated(subset=["basin_id"], keep=False)].sort_values(by=list(basins_gdf.columns))

In [ ]:
zones_gdf = gpd.read_parquet("rfc_zones.parquet")
zones_gdf.head()

In [ ]:
zones_gdf[zones_gdf.zone_id == "close"]

In [ ]:
# This is a duplicate of a basin layer polygon
zones_gdf.loc[3706, "zone_id"] = "NCA7"

In [ ]:
basin_ids = basins_gdf.basin_id.tolist()

Remove features (zone IDs) in the zones layer that also exist in the `basins_gdf` layer (basin ID)

In [ ]:
zones_only_gdf = zones_gdf[~zones_gdf["zone_id"].isin(basin_ids)]

Load both of these with the same prefix (`rfcbasins-`)?

In [ ]:
zone_only_ids = zones_only_gdf.zone_id.tolist()

NOTE: Some of the IDs are duplicated across RFCs, include the RFC in the `location_id`

In [ ]:
zones_only_gdf.loc[:, "location_id"] = zones_only_gdf['zone_id'].astype(str) + ':' + zones_only_gdf['RFC'].astype(str)
basins_gdf.loc[:, "location_id"] = basins_gdf['basin_id'].astype(str) + ':' + basins_gdf['RFC'].astype(str)

Make sure no duplicates exist

In [ ]:
zones_only_gdf[zones_only_gdf.duplicated(subset=["location_id"], keep=False)]

Now check to see if there are duplicates across both sets of IDs

In [ ]:
zone_location_ids = zones_only_gdf.location_id.tolist()
basin_location_ids = basins_gdf.location_id.tolist()

all_ids = zone_location_ids + basin_location_ids

In [ ]:
len(all_ids), len(set(all_ids))

In [ ]:
zones_only_gdf.to_parquet("~/rfc_zones_only.parquet")

In [ ]:
zones_only_gdf.crs

In [ ]:
basins_gdf.crs

#### Fix any invalid geometry

In [ ]:
%%time
basins_gdf["geometry"] = basins_gdf.geometry.buffer(0)
zones_only_gdf["geometry"] = zones_only_gdf.geometry.buffer(0)

In [ ]:
# Save for later so we can skip previous step next time
basins_gdf.to_parquet("rfc_basins_valid.parquet")
zones_only_gdf.to_parquet("rfc_zones_valid.parquet")

In [ ]:
basins_gdf.head()

In [ ]:
zones_only_gdf.head()

Add and populate the name column

In [ ]:
zones_only_gdf.loc[:, "name"] = zones_only_gdf.RFC.astype(str) + " elevation zone " + zones_only_gdf.zone_id.astype(str)
zones_only_gdf.head()

In [ ]:
basins_gdf.loc[:, "name"] = basins_gdf.RFC.astype(str) + " basin " + basins_gdf.basin_id.astype(str)
basins_gdf.head()

Assign prefix: Same for both basins and zones. Save as separate layers in case there is some use there.

In [ ]:
basins_gdf["location_id"] = "rfcbasin-" + basins_gdf["location_id"]
basins_gdf.head()

In [ ]:
zones_only_gdf["location_id"] = "rfczone-" + zones_only_gdf["location_id"]
zones_only_gdf.head()

Assign properties with an estimate on when they were pulled from the FEWS configs

In [ ]:
props = {"source": "FEWS configuration", "accessed": "05/21/2026"}

In [ ]:
zones_only_gdf["properties"] = [props.copy() for _ in range(len(zones_only_gdf))]
zones_only_gdf.head()

In [ ]:
basins_gdf["properties"] = [props.copy() for _ in range(len(basins_gdf))]
basins_gdf.head()

In [ ]:
zones_only_gdf[["location_id", "name", "geometry", "properties"]].to_parquet("rfc_zones_teehr.parquet")
basins_gdf[["location_id", "name", "geometry", "properties"]].to_parquet("rfc_basins_teehr.parquet")

#### Let's combine into a single layer

In [ ]:
basins = gpd.read_parquet("rfc_basins_teehr.parquet")
zones = gpd.read_parquet("rfc_zones_teehr.parquet")
basins.head()

In [ ]:
zones.head()

In [ ]:
zones["location_id"] = zones["location_id"].str.replace(r"rfczone-", "rfcbasin-", regex=True)
zones.head()

In [ ]:
combined = pd.concat([zones, basins])
combined.head()

In [ ]:
combined.rename(columns={"location_id": "id"}, inplace=True)
combined.head()

In [ ]:
combined.to_parquet("rfc_combined_teehr.parquet", index=False)

In [ ]:
import shutil

shutil.copy2("rfc_combined_teehr.parquet", "/data/common/geometry/rfc_basins/rfc_combined_teehr.parquet")